# scrna-toolkit: basic usage

load → harmonise metadata → QC → filter → preserve counts → normalize → HVGs → PCA → neighbors → UMAP → summary

This notebook uses a small **synthetic** dataset so that it runs anywhere. Replace the first cell with your own data, e.g. `sct.load_scrna("filtered_feature_bc_matrix.h5")`. The QC thresholds below are illustrations for the toy data, not recommendations.

In [ ]:
import logging
import tempfile
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

import scrna_toolkit as sct

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

In [ ]:
# Toy data: 3 cell populations x 4 samples from 2 donors (written to a temporary h5ad file).
rng = np.random.default_rng(0)
n_cells, n_genes = 600, 500
population = rng.integers(0, 3, n_cells)
means = rng.gamma(1.0, 1.0, size=(3, n_genes))
counts = rng.poisson(means[population]).astype(np.float32)
genes = [f"MT-G{i}" for i in range(10)] + [f"GENE{i}" for i in range(n_genes - 10)]
obs = pd.DataFrame(
    {
        "orig.ident": [f"S{i % 4}" for i in range(n_cells)],
        "donor": [f"D{i % 2}" for i in range(n_cells)],
        "annotation": np.array(["T", "B", "Myeloid"])[population],
    },
    index=[f"cell{i}" for i in range(n_cells)],
)
path = Path(tempfile.mkdtemp()) / "toy.h5ad"
ad.AnnData(sp.csr_matrix(counts), obs=obs, var=pd.DataFrame(index=genes)).write_h5ad(path)

## Load

In [ ]:
adata = sct.load_scrna(path)  # format inferred; also 10x .h5, 10x MTX dirs, CSV/TSV
adata

## Harmonise metadata
Copy dataset-specific columns to canonical names (originals are kept), then check for problems.

In [ ]:
sct.harmonise_obs(
    adata, {"orig.ident": "sample_id", "donor": "patient_id", "annotation": "cell_type"}
)
sct.validate_metadata(adata).query("severity != 'info'")  # empty = no problems

## QC

In [ ]:
sct.calculate_qc_metrics(adata, mitochondrial_prefix="MT-")  # raises if X is not raw counts
sct.qc_summary(adata)

In [ ]:
sct.plot_qc_distributions(adata, groupby="sample_id");

## Filter
Filters return a new object; the summary shows how many cells fail each criterion.

In [ ]:
adata, summary = sct.filter_cells(adata, min_genes=235, max_pct_mt=5, return_summary=True)
adata = sct.filter_genes(adata, min_cells=3)
summary

## Preserve counts and normalize
`layers["counts"]` keeps the raw counts; `X` becomes log-normalized.

In [ ]:
sct.store_counts(adata)
sct.normalize_total(adata, target_sum=1e4)
sct.log1p(adata)
sct.ensure_raw_counts(adata, layer="counts")  # raises if the stored counts were altered
sct.get_representation(adata)  # what adata.X holds now

## HVGs, PCA, neighbors, UMAP, clustering
The same steps as `sct.preprocess(adata, n_hvgs=..., n_pcs=...)`, run one by one.

In [ ]:
sct.select_hvgs(adata, n_top_genes=300)
sct.run_pca(adata, n_comps=20)  # PCA on a scaled copy of the HVGs; X is untouched
sct.run_neighbors(adata, n_neighbors=15)
sct.run_umap(adata)
sct.run_leiden(adata, resolution=0.5);

In [ ]:
sct.plot_umap(adata, color=["cell_type", "patient_id", "leiden"]);

## Summaries

In [ ]:
sct.summarize_dataset(adata)

In [ ]:
sct.summarize_by(adata, "patient_id")

In [ ]:
sct.plot_cell_type_composition(adata, groupby="sample_id");

## Provenance
Every step and its parameters are stored in `adata.uns["scrna_toolkit"]` and survive `write_h5ad`.

In [ ]:
pd.DataFrame(sct.get_history(adata))[["step", "params"]]